# LC 297 — Serialize and Deserialize Binary Tree
**Difficulty:** Hard | **Pattern:** BFS Level-Order Traversal

<div style="border-left:4px solid purple; padding:10px 16px;
background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> BFS level-order traversal naturally
captures the complete tree structure — including None children —
in a flat string. Deserializing reverses this exactly: a queue
tracks parent nodes waiting for children, and you assign left
then right from the token list in the same BFS order.
</div>

## Official Problem Statement

Serialization is the process of converting a data structure or
object into a sequence of bits so that it can be stored in a file
or memory buffer, or transmitted across a network connection link
to be reconstructed later in the same or another computer
environment.

Design an algorithm to serialize and deserialize a binary tree.
There is no restriction on how your serialization/deserialization
algorithm should work. You just need to ensure that a binary tree
can be serialized to a string and this string can be deserialized
to the original tree structure.

**Constraints:**
- The number of nodes in the tree is in the range `[0, 10^4]`.
- `-1000 <= Node.val <= 1000`
- The encoded string should not contain spaces (recommended).
- Implement `serialize(root)` and `deserialize(data)` as a class.

## What This Is Actually Asking

You need to flatten a binary tree into a string and then rebuild
the exact same tree from that string — a complete round-trip.

The challenge is encoding *structure*, not just values: you must
know where None children are so you can reconstruct the shape.

BFS level-order is ideal because it processes nodes parent-before-
children, so during deserialization the queue-based reconstruction
mirrors the queue-based serialization perfectly.

This problem has no single "right" answer — any format that enables
a faithful round-trip is valid. BFS is the most intuitive.

## Walk Through an Example by Hand

```
Tree:      1
          / \
         2   3
            / \
           4   5
```

**Serialize (BFS level-order):**
```
Queue: [1]
  pop 1  → output "1",  enqueue 2, 3
Queue: [2, 3]
  pop 2  → output "2",  enqueue null, null
  pop 3  → output "3",  enqueue 4, 5
Queue: [None, None, 4, 5]
  pop None → output "null"
  pop None → output "null"
  pop 4  → output "4",  enqueue null, null
  pop 5  → output "5",  enqueue null, null

Result: "1,2,3,null,null,4,5"
```

**Deserialize:**
```
tokens = ["1","2","3","null","null","4","5"]
root = TreeNode(1); queue = [node(1)]; i = 1
  pop node(1): left=node(2) [i=2], right=node(3) [i=3]
  pop node(2): left=null   [i=4], right=null  [i=5]
  pop node(3): left=node(4)[i=6], right=node(5)[i=7]
  (4 and 5 have no children left in tokens)
Rebuilt tree matches original.
```

## The Picture

```
SERIALIZE — BFS with None markers:

    Level 0:        1          → "1"
    Level 1:      2   3        → "2,3"
    Level 2:  N N  4   5       → "null,null,4,5"
    Level 3: (all null)        → omit trailing nulls

    String: "1,2,3,null,null,4,5"
             ↑ ↑ ↑  ↑    ↑  ↑ ↑
             0 1 2  3    4  5 6  ← token indices

DESERIALIZE — BFS reconstruction:

    Parent queue:     Children assigned from token list:
    ┌──────────┐
    │  node(1) │ → left  = tokens[1] = "2"    → node(2)
    │          │   right = tokens[2] = "3"    → node(3)
    └──────────┘
    ┌──────────┐
    │  node(2) │ → left  = tokens[3] = "null" → None
    │          │   right = tokens[4] = "null" → None
    └──────────┘
    ┌──────────┐
    │  node(3) │ → left  = tokens[5] = "4"    → node(4)
    │          │   right = tokens[6] = "5"    → node(5)
    └──────────┘

    KEY: Only real nodes (non-null) are added to queue as parents.
         Each parent always consumes exactly 2 tokens (left, right).
```

## When To Use This Pattern

- When you need to **persist or transmit** a tree structure and
  recreate it exactly, think **BFS serialization with null markers**.

- When the problem requires encoding both **values and structure**
  (shape of the tree), think **level-order with explicit nulls**.

- When rebuilding a data structure from a flat representation,
  think **queue-based BFS reconstruction** — parent node in queue
  consumes the next two tokens as its children.

- When you see serialize/deserialize for any hierarchical data
  (trees, tries, nested JSON), think **traversal order + delimiters
  + sentinel values for missing children**.

- When comparing two trees for equality after a round-trip, think
  **recursive structural comparison** node by node.

## The Approach

For serialization, run a standard BFS using a queue. For each node
dequeued, append its value (or `"null"`) to the output list, and
enqueue both children (including None children so their parents
record them). Join the list with commas.

For deserialization, split on commas to get a token list. Create
the root from index 0 and seed a queue with it. For each node
dequeued, assign the next two tokens as its left and right children;
skip assignment (leave as None) for `"null"` tokens, and only add
real nodes to the queue as future parents.

The BFS order guarantees that serialization and deserialization
consume tokens in exactly the same sequence, making the round-trip
faithful.

In [ ]:
from typing import List
from collections import defaultdict, deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def make_tree(vals):
    """Build a tree from level-order list (None = missing node)."""
    if not vals:
        return None
    root = TreeNode(vals[0])
    q = deque([root])
    i = 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            q.append(node.right)
        i += 1
    return root


def tree_to_list(root):
    """Convert tree back to level-order list for comparison."""
    if not root:
        return []
    result = []
    q = deque([root])
    while q:
        node = q.popleft()
        if node:
            result.append(node.val)
            q.append(node.left)
            q.append(node.right)
        else:
            result.append(None)
    while result and result[-1] is None:
        result.pop()
    return result

In [ ]:
def test_harness(codec_class):
    """
    Test harness for Codec serialize/deserialize.
    Round-trip: serialize → deserialize → compare tree structure.
    """
    tests = [
        (
            [1, 2, 3, None, None, 4, 5],
            "classic LeetCode example"
        ),
        (
            [],
            "empty tree"
        ),
        (
            [1],
            "single node"
        ),
        (
            [1, 2],
            "left child only"
        ),
        (
            [1, None, 2, None, 3],
            "right-skewed tree"
        ),
        (
            [-1000, 1000, -1000],
            "boundary values"
        ),
    ]

    passed = 0
    codec = codec_class()
    for vals, desc in tests:
        original = make_tree(vals)
        serialized = codec.serialize(original)
        rebuilt = codec.deserialize(serialized)
        orig_list = tree_to_list(original)
        rebuilt_list = tree_to_list(rebuilt)
        ok = (orig_list == rebuilt_list)
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(
            f"[{status}] {desc}"
            f" | data='{serialized}'"
        )

    print(f"\n{passed}/{len(tests)} tests passed.")

In [ ]:
class Codec:
    """
    Serialize and deserialize a binary tree using BFS level-order.

    serialize(root) -> str:
        BFS traversal; include 'null' for missing children.
        Join tokens with ',' and return the string.
        Returns '' for an empty tree.

    deserialize(data) -> TreeNode:
        Split data by ','. Build root from index 0.
        Use a queue of parent nodes. For each parent, consume
        next two tokens as left and right children.
        Only enqueue non-null children as future parents.
        Returns None for empty string input.

    Time:  O(N) for both serialize and deserialize.
    Space: O(N) for queue and token list.

    Debug prints show serialized string and token list.
    """

    def serialize(self, root: TreeNode) -> str:
        """Encode tree to a single string via BFS."""
        if not root:
            print("[DEBUG] Empty tree → ''")
            return ""

        tokens = []
        queue = deque([root])

        while queue:
            node = queue.popleft()
            if node:
                tokens.append(str(node.val))
                queue.append(node.left)
                queue.append(node.right)
            else:
                tokens.append("null")

        result = ",".join(tokens)
        print(f"[DEBUG] serialized: '{result}'")
        return result

    def deserialize(self, data: str) -> TreeNode:
        """Decode string back to tree via BFS reconstruction."""
        if not data:
            print("[DEBUG] Empty string → None")
            return None

        tokens = data.split(",")
        print(f"[DEBUG] tokens: {tokens}")

        root = TreeNode(int(tokens[0]))
        queue = deque([root])
        i = 1  # pointer into token list

        while queue and i < len(tokens):
            node = queue.popleft()

            # Assign left child
            if i < len(tokens) and tokens[i] != "null":
                node.left = TreeNode(int(tokens[i]))
                queue.append(node.left)
            i += 1

            # Assign right child
            if i < len(tokens) and tokens[i] != "null":
                node.right = TreeNode(int(tokens[i]))
                queue.append(node.right)
            i += 1

        return root

In [ ]:
# Uncomment and run when solution is ready
# test_harness(Codec)

## Complexity

| Approach | Time | Space | Notes |
|---|---|---|---|
| Brute force (naive recursion, no nulls) | O(N²) | O(N) | Ambiguous for reconstruction |
| BFS level-order with null markers | O(N) | O(N) | Unambiguous round-trip |
| DFS preorder with null markers | O(N) | O(H) | H = tree height |

**N** = number of nodes. BFS serializes exactly N real nodes and
at most N+1 null markers, so both serialize and deserialize run
in O(N) time and O(N) space for the queue and token list.

## Real World Connection

At **Citi**, trade blotter hierarchies (accounts → sub-accounts
→ positions) must be serialized to JSON or Avro for messaging
between systems — exactly this pattern, where null children must
be explicitly encoded to preserve structural fidelity.

**AWS DynamoDB** stores nested document structures as serialized
attribute maps; the SDK internally converts tree-like Python dicts
to DynamoDB's wire format and back, a conceptual serialize/
deserialize round-trip.

In **data engineering**, decision trees in ML pipelines (e.g.,
scikit-learn's `export_graphviz` or XGBoost model dumps) are
serialized tree structures — saving a trained model is exactly
serialization, and loading it is deserialization.

Understanding BFS-based serialization is foundational for working
with any system that checkpoints, replicates, or transmits
hierarchical state — a constant in distributed data systems.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra